# Qwen3-Omni

In [1]:
import os
import json
import tempfile
from pathlib import Path

import librosa
import soundfile as sf
import pandas as pd
from tqdm import tqdm
from sklearn.metrics import accuracy_score, f1_score
from transformers import Qwen3OmniMoeForConditionalGeneration, Qwen3OmniMoeProcessor
from qwen_omni_utils import process_mm_info



/root/miniconda3/envs/qwen/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
!source /etc/network_turbo

设置成功
注意：仅限于学术用途和加速访问github/huggingface，不承诺稳定性保证


In [3]:
CACHE_DIR    = "/root/autodl-tmp/LLM_Model"
PROJECT_ROOT = Path("/root/autodl-tmp/Few-Shot_is_all_you_need/ad_detection")
MODEL_ID     = "Qwen/Qwen3-Omni-30B-A3B-Instruct"

os.environ["HF_HOME"] = CACHE_DIR

model = Qwen3OmniMoeForConditionalGeneration.from_pretrained(
    MODEL_ID,
    torch_dtype="auto",
    device_map="auto",
    attn_implementation="flash_attention_2",
    cache_dir=CACHE_DIR,
)
model.disable_talker()

processor = Qwen3OmniMoeProcessor.from_pretrained(MODEL_ID, cache_dir=CACHE_DIR)

print("Qwen3-Omni model loaded.")

`torch_dtype` is deprecated! Use `dtype` instead!
Unrecognized keys in `rope_scaling` for 'rope_type'='default': {'mrope_interleaved', 'mrope_section', 'interleaved'}
Unrecognized keys in `rope_scaling` for 'rope_type'='default': {'mrope_section', 'interleaved'}
You are attempting to use Flash Attention 2 without specifying a torch dtype. This might lead to unexpected behaviour
Loading checkpoint shards: 100%|██████████| 15/15 [00:15<00:00,  1.02s/it]
Some parameters are on the meta device because they were offloaded to the cpu.
The image processor of type `Qwen2VLImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. Note that this behavior will be extended to all models in a future release.


Qwen3-Omni model loaded.


In [4]:
SYSTEM_PROMPT = (
    "You are a clinical speech-language pathologist specialized in detecting "
    "Alzheimer's disease and dementia from spontaneous speech. You analyze speech "
    "patterns including: word-finding difficulties, semantic paraphasias, empty speech, "
    "reduced syntactic complexity, repetitions, incomplete utterances, and pragmatic "
    "impairments. Based on the audio, classify the speaker."
)

USER_PROMPT = (
    "Listen to this speech sample carefully. Based on the speech characteristics, "
    "is this speaker showing signs of dementia or is this a healthy control? "
    "Answer with exactly one word: 'Dementia' or 'Control'."
)

In [5]:
TMP_WAV_DIR = Path(tempfile.mkdtemp(prefix="qwen3_wav_"))


def ensure_wav(audio_path: Path) -> Path:
    """Convert mp3 to 16kHz mono wav via librosa if needed."""
    if audio_path.suffix.lower() == ".wav":
        return audio_path
    wav_path = TMP_WAV_DIR / f"{audio_path.stem}.wav"
    if not wav_path.exists():
        audio, sr = librosa.load(str(audio_path), sr=16000, mono=True)
        sf.write(str(wav_path), audio, sr)
    return wav_path

In [6]:
VALID_LABELS = {"Dementia", "Control"}
USE_AUDIO_IN_VIDEO = True


def classify_audio(wav_path: Path) -> str:
    """Classify a single audio file. Returns raw model response."""
    conversation = [
        {"role": "system", "content": [{"type": "text", "text": SYSTEM_PROMPT}]},
        {
            "role": "user",
            "content": [
                {"type": "audio", "audio": str(wav_path)},
                {"type": "text",  "text": USER_PROMPT},
            ],
        },
    ]

    text = processor.apply_chat_template(conversation, add_generation_prompt=True, tokenize=False)
    audios, images, videos = process_mm_info(conversation, use_audio_in_video=USE_AUDIO_IN_VIDEO)
    inputs = processor(
        text=text,
        audio=audios,
        images=images,
        videos=videos,
        return_tensors="pt",
        padding=True,
        use_audio_in_video=USE_AUDIO_IN_VIDEO,
    )
    inputs = inputs.to(model.device).to(model.dtype)

    text_ids, _ = model.generate(
        **inputs,
        return_audio=False,
        use_audio_in_video=USE_AUDIO_IN_VIDEO,
        max_new_tokens=64,
    )
    output = processor.batch_decode(
        text_ids[:, inputs["input_ids"].shape[1]:],
        skip_special_tokens=True,
        clean_up_tokenization_spaces=False,
    )
    return output[0]

In [7]:
def evaluate_dataset(csv_path, audio_dir, name=""):
    df = pd.read_csv(csv_path)
    label_map = {0: "Control", 1: "Dementia"}
    predictions, skipped = [], 0

    audio_dir = Path(audio_dir)
    print(f"[{name}] audio_dir={audio_dir}, exists={audio_dir.exists()}")

    for idx, (_, row) in enumerate(tqdm(df.iterrows(), total=len(df), desc=name)):
        label_dir = label_map[row["ad"]]
        matches = list(audio_dir.glob(f"{label_dir}/{row['session_id']}.*"))
        if not matches:
            skipped += 1
            continue
        try:
            raw = classify_audio(ensure_wav(matches[0]))
            pred = raw.strip() if raw.strip() in VALID_LABELS else None
        except Exception as e:
            raw, pred = str(e), None
        if idx < 3:
            print(f"  DEBUG [{idx}] session={row['session_id']} raw={repr(raw[:200])} pred={pred}")
        predictions.append({"session_id": row["session_id"], "true": label_dir, "pred": pred, "raw": raw})

    valid = [p for p in predictions if p["pred"] is not None]
    y_true = [p["true"] for p in valid]
    y_pred = [p["pred"] for p in valid]
    n, total = len(valid), len(df)
    ctrl = [p for p in valid if p["true"] == "Control"]
    dem  = [p for p in valid if p["true"] == "Dementia"]

    print(f"[{name}]")
    print(f"  Accuracy:    {accuracy_score(y_true, y_pred):.4f}")
    print(f"  F1:          {f1_score(y_true, y_pred, pos_label='Dementia'):.4f}")
    print(f"  Control Acc: {sum(p['pred']=='Control'  for p in ctrl)/max(len(ctrl),1):.4f}")
    print(f"  Dementia Acc:{sum(p['pred']=='Dementia' for p in dem) /max(len(dem),1) :.4f}")
    print(f"  Valid: {n}/{total}  Skipped: {skipped}")

In [8]:
# import sys; sys.path.insert(0, str(PROJECT_ROOT / "train"))
# from data_split import create_test_csv

# csv       = PROJECT_ROOT / "data/processed/Pitt-xlsr-test.csv"
# if not csv.exists() or csv.stat().st_size < 30:
#     create_test_csv(PROJECT_ROOT / "data/raw/Pitt", "Pitt", "Pitt_xlsr_features", xlsr=True)
    
# audio_dir = PROJECT_ROOT / "data/raw/Pitt"
# evaluate_dataset(csv, audio_dir, "Pitt-raw")

In [9]:
# csv       = PROJECT_ROOT / "data/processed/Lu-xlsr-test.csv"
# if not csv.exists() or csv.stat().st_size < 30:
#     create_test_csv(PROJECT_ROOT / "data/raw/Lu", "Lu", "Lu_xlsr_features", xlsr=True)
    
# audio_dir = PROJECT_ROOT / "data/raw/Lu"
# evaluate_dataset(csv, audio_dir, "Lu-raw")

In [10]:
# csv       = PROJECT_ROOT / "data/processed/Pitt-xlsr-test.csv"
# audio_dir = PROJECT_ROOT / "data/denoised/Pitt-Demucs"
# evaluate_dataset(csv, audio_dir, "Pitt-Demucs")

In [11]:
# csv       = PROJECT_ROOT / "data/processed/Lu-xlsr-test.csv"
# audio_dir = PROJECT_ROOT / "data/denoised/Lu-Demucs"
# evaluate_dataset(csv, audio_dir, "Lu-Demucs")

In [ ]:
csv       = PROJECT_ROOT / "data/processed/Pitt-xlsr-test.csv"
audio_dir = PROJECT_ROOT / "data/denoised/Pitt-Denoiser"
evaluate_dataset(csv, audio_dir, "Pitt-Denoiser")

In [ ]:
# csv       = PROJECT_ROOT / "data/processed/Lu-xlsr-test.csv"
# audio_dir = PROJECT_ROOT / "data/denoised/Lu-Denoiser"
# evaluate_dataset(csv, audio_dir, "Lu-Denoiser")

[Lu-Denoiser] audio_dir=/root/autodl-tmp/Few-Shot_is_all_you_need/ad_detection/data/denoised/Lu-Denoiser, exists=True


Lu-Denoiser:   1%|▏         | 1/74 [00:12<15:33, 12.79s/it]Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.


  DEBUG [0] session=F22_000 raw='Dementia' pred=Dementia


Lu-Denoiser:   3%|▎         | 2/74 [00:22<12:54, 10.76s/it]Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.


  DEBUG [1] session=F22_001 raw='Dementia' pred=Dementia


Lu-Denoiser:   4%|▍         | 3/74 [00:31<11:55, 10.08s/it]Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.


  DEBUG [2] session=F26_000 raw='Dementia' pred=Dementia


Lu-Denoiser: 100%|██████████| 74/74 [11:00<00:00,  8.93s/it]

[Lu-Denoiser]
  Accuracy:    0.5541
  F1:          0.6916
  Control Acc: 0.1111
  Dementia Acc:0.9737
  Valid: 74/74  Skipped: 0


In [ ]:
csv       = PROJECT_ROOT / "data/processed/Pitt-xlsr-test.csv"
audio_dir = PROJECT_ROOT / "data/denoised/Pitt-FRCRN_SE"
evaluate_dataset(csv, audio_dir, "Pitt-FRCRN_SE")

In [ ]:
# csv       = PROJECT_ROOT / "data/processed/Lu-xlsr-test.csv"
# audio_dir = PROJECT_ROOT / "data/denoised/Lu-FRCRN_SE"
# evaluate_dataset(csv, audio_dir, "Lu-FRCRN_SE")

[Lu-FRCRN_SE] audio_dir=/root/autodl-tmp/Few-Shot_is_all_you_need/ad_detection/data/denoised/Lu-FRCRN_SE, exists=True


Lu-FRCRN_SE:   1%|▏         | 1/74 [00:08<10:43,  8.82s/it]Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.


  DEBUG [0] session=F22_000 raw='Dementia' pred=Dementia


Lu-FRCRN_SE:   3%|▎         | 2/74 [00:17<10:41,  8.91s/it]Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.


  DEBUG [1] session=F22_001 raw='Dementia' pred=Dementia


Lu-FRCRN_SE:   4%|▍         | 3/74 [00:26<10:31,  8.89s/it]Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.


  DEBUG [2] session=F26_000 raw='Dementia' pred=Dementia


Lu-FRCRN_SE: 100%|██████████| 74/74 [10:52<00:00,  8.82s/it]

[Lu-FRCRN_SE]
  Accuracy:    0.5405
  F1:          0.6792
  Control Acc: 0.1111
  Dementia Acc:0.9474
  Valid: 74/74  Skipped: 0


In [ ]:
csv       = PROJECT_ROOT / "data/processed/Pitt-xlsr-test.csv"
audio_dir = PROJECT_ROOT / "data/denoised/Pitt-MossFormer"
evaluate_dataset(csv, audio_dir, "Pitt-MossFormer")

In [ ]:
# csv       = PROJECT_ROOT / "data/processed/Lu-xlsr-test.csv"
# audio_dir = PROJECT_ROOT / "data/denoised/Lu-MossFormer"
# evaluate_dataset(csv, audio_dir, "Lu-MossFormer")

[Lu-MossFormer] audio_dir=/root/autodl-tmp/Few-Shot_is_all_you_need/ad_detection/data/denoised/Lu-MossFormer, exists=True


Lu-MossFormer:   1%|▏         | 1/74 [00:08<10:46,  8.86s/it]Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.


  DEBUG [0] session=F22_000 raw='Dementia' pred=Dementia


Lu-MossFormer:   3%|▎         | 2/74 [00:17<10:39,  8.89s/it]Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.


  DEBUG [1] session=F22_001 raw='Dementia' pred=Dementia


Lu-MossFormer:   4%|▍         | 3/74 [00:26<10:27,  8.84s/it]Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.


  DEBUG [2] session=F26_000 raw='Dementia' pred=Dementia


Lu-MossFormer: 100%|██████████| 74/74 [11:02<00:00,  8.95s/it]

[Lu-MossFormer]
  Accuracy:    0.5676
  F1:          0.7037
  Control Acc: 0.1111
  Dementia Acc:1.0000
  Valid: 74/74  Skipped: 0


In [ ]:
csv       = PROJECT_ROOT / "data/processed/Pitt-xlsr-test.csv"
audio_dir = PROJECT_ROOT / "data/denoised/Pitt-Resemble"
evaluate_dataset(csv, audio_dir, "Pitt-Resemble")

In [19]:
csv       = PROJECT_ROOT / "data/processed/Lu-xlsr-test.csv"
audio_dir = PROJECT_ROOT / "data/denoised/Lu-Resemble"
evaluate_dataset(csv, audio_dir, "Lu-Resemble")

[Lu-Resemble] audio_dir=/root/autodl-tmp/Few-Shot_is_all_you_need/ad_detection/data/denoised/Lu-Resemble, exists=True


Lu-Resemble:   1%|▏         | 1/74 [00:09<10:59,  9.03s/it]Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.


  DEBUG [0] session=F22_000 raw='Dementia' pred=Dementia


Lu-Resemble:   3%|▎         | 2/74 [00:18<10:59,  9.16s/it]Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.


  DEBUG [1] session=F22_001 raw='Dementia' pred=Dementia


Lu-Resemble:   4%|▍         | 3/74 [00:27<10:47,  9.12s/it]Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.


  DEBUG [2] session=F26_000 raw='Dementia' pred=Dementia


Lu-Resemble: 100%|██████████| 74/74 [10:58<00:00,  8.90s/it]

[Lu-Resemble]
  Accuracy:    0.5405
  F1:          0.6852
  Control Acc: 0.0833
  Dementia Acc:0.9737
  Valid: 74/74  Skipped: 0
